# Figure 2C Stage-associated expression heatmap

Start from the saved embryonic RNA object. Keep the integration, original annotations and Slingshot model unchanged. This notebook performs stage-associated differential expression, exploratory marker selection and heatmap plotting.

Formal comparisons use three biological replicates per stage in GSE195500 at E14.5, E16.5 and E18.5. Each target stage is compared with the geometric mean of fitted expression at the other two stages, using TMM, edgeR quasi-likelihood, glmTreat and joint BH correction. Other stage comparisons are exploratory and do not remove source or state-composition confounding.


In [ ]:
#1. =========== Paths and shared settings
from pathlib import Path
import json
import hashlib
import os
import subprocess
import tempfile

# Replace each ROOT placeholder with the required file or directory path before running.
ROOT = Path(".")
if ROOT.name == "Code_def":
    ROOT = ROOT.parent
# Uses CSS_3000_Embryonic_Inte_adata.h5ad generated by fig1_Embryonic_integration.ipynb.
RNA_FILE = ROOT
# Set the file or directory path.
DATA_DIR = ROOT
# Set the file or directory path.
FIG_DIR = ROOT
# Set the file or directory path.
TMP_DIR = ROOT
# Uses Stage-heatmap.h5ad generated by fig2_embry_heatmap.ipynb.
SUMMARY_FILE = ROOT
# Uses Stage-DEG.tsv.gz generated by fig2_embry_heatmap.ipynb.
DEG_FILE = ROOT
# Set the executable or tool directory path.
RSCRIPT = ROOT
SEED, N_THREADS = 0, 1
STAGE, STATE, STUDY = "stage_standardized", "cell_type_plot", "study_id"
STAGES = ["E9.5", "E10.5", "E11.5", "E13.5", "E14.5", "E15.5", "E16.5", "E17.5", "E18.5"]
STAGE_COLORS = dict(zip(STAGES, ["#D157A0", "#CC0744", "#3B9700", "#3B5DFF", "#D16100", "#00846F", "#BC23FF", "#6A3A4C", "#9B9700"]))
STUDY_COLORS = {"GSE178931": "#829FB3", "GSE231513": "#C3A580", "GSE195500": "#758677", "GSE165502": "#A493B2"}
STATE_ORDER = ["CVG neuroblast", "O-2: delaminating otic/CVG neuroblast", "O-3: developing CVG neuron (not SGN-specific)", "Undifferentiated / proliferative CVG", "Late CVG", "Differentiating SGN", "Undifferentiated SGN", "Type Ia + Type II precursor candidate", "Type Ib + Type Ic precursor candidate", "Emerging Type Ic", "Immature Type Ia SGN", "Immature Type Ib SGN", "Immature Type Ic SGN", "Immature Type II SGN"]
CVG_STATES = STATE_ORDER[:5]
KNOWN_GENES = ["Insm1", "Neurod1", "Sox2", "Myt1", "Gata3", "Mafb", "Calb1", "Calb2", "Lypd1", "Ntng1", "Th", "Plk5", "Espn", "Tac1", "Shox2", "Casz1", "Id1", "Eya1", "Six1", "Neurog1", "Runx1", "Pou4f1", "Prph", "Sall3", "Zeb1", "Tle4", "Efna5"]

TMP_DIR.mkdir(parents=True, exist_ok=True)
# Set the file or directory path.
os.environ.update(OMP_NUM_THREADS=str(N_THREADS), OPENBLAS_NUM_THREADS=str(N_THREADS), MKL_NUM_THREADS=str(N_THREADS), MPLCONFIGDIR=str(ROOT), TMP=str(TMP_DIR), TEMP=str(TMP_DIR))
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_rgba
from matplotlib.patches import Patch

from matplotlib.collections import QuadMesh
from PIL import Image as PILImage
import win32com.client.dynamic as com
from IPython.display import display, Image

np.random.seed(SEED)
REBUILD_RESULTS = False  # True: rerun from the integration RNA; False: reuse the two key result files.

## RNA summaries

Reuse `X=log1p(CP10k)` without renormalization; formal tests use `layers["counts"]`. Keep `source_state_evaluation_allowed` and `cell_type_plot`. Display columns are sample/library-by-state means with at least three cells.


In [ ]:
#2. =========== RNA summaries
MIN_DISPLAY_CELLS = 3                 # Hide sparsely sampled state means while retaining those cells for DE.
MIN_DETECTION = 0.10                 # Target-stage detection fraction.
MIN_DETECTED_CELLS = 3               # Minimum detected cells per supporting unit.
MIN_UNIT_SUPPORT = 2 / 3             # Direction and detection agreement across units.
RATIO_PSEUDOCOUNT = 0.1              # CP10k stabilizer for exploratory ratios only.
MIN_FORMAL_CELLS = 20                # Minimum SGN cells per biological replicate.


def unit_summary(obs, linear, log_rna, counts, keys):
    rows, means, logs, fractions, detected, sums = [], [], [], [], [], []
    for key, ids in obs.groupby(keys, observed=True, sort=False).indices.items():
        key = key if isinstance(key, tuple) else (key,)
        rows.append(dict(zip(keys, key), n_cells=len(ids)))
        means.append(np.asarray(linear[ids].mean(axis=0)).ravel())
        logs.append(np.asarray(log_rna[ids].mean(axis=0)).ravel())
        nd = np.asarray((counts[ids] > 0).sum(axis=0)).ravel()
        detected.append(nd)
        fractions.append(nd / len(ids))
        sums.append(np.asarray(counts[ids].sum(axis=0)).ravel())
    return pd.DataFrame(rows), np.array(means), np.array(logs), np.array(fractions), np.array(detected), np.array(sums)


def balance_by_stage(meta, matrix, stages):
    frame = pd.DataFrame(matrix)
    frame[STAGE], frame[STUDY] = meta[STAGE].to_numpy(), meta[STUDY].to_numpy()
    return frame.groupby([STAGE, STUDY], observed=True).mean().groupby(level=0).mean().reindex(stages).to_numpy()


def build_summaries():
    a = ad.read_h5ad(RNA_FILE)
    obs = a.obs.loc[a.obs.source_state_evaluation_allowed.astype(bool)].copy()
    assert len(obs) == 6995 and set(obs[STAGE]) == set(STAGES)
    obs["compartment"] = np.where(obs[STATE].isin(CVG_STATES), "CVG", "SGN")
    obs["unit"] = obs.biological_replicate_id.astype(str).where(obs.biological_replicate_id.astype(str) != "Not available", obs.library_id.astype(str))
    obs["unit_kind"] = np.where(obs[STUDY] == "GSE195500", "biological_replicate", "library_or_plate")
    for field in [STAGE, STATE, STUDY]:
        obs[field] = obs[field].astype(str)
    ids = a.obs_names.get_indexer(obs.index)
    counts, log_rna = a.layers["counts"][ids].tocsr(), a.X[ids].tocsr()
    assert np.issubdtype(counts.dtype, np.integer) and counts.min() >= 0
    linear = log_rna.copy()
    linear.data = np.expm1(linear.data)
    assert np.allclose(np.asarray(linear.sum(axis=1)).ravel(), 1e4, rtol=2e-4)
    genes = a.var[["gene_symbol", "mgi_id", "chromosome"]].copy()
    genes.index.name = "gene_id"
    genes["eligible_rna"] = ~(genes.chromosome.str.lower().isin(["chrm", "chrmt", "mt", "m"]) | genes.gene_symbol.str.contains(r"^mt-|^ERCC[-_]", case=False, regex=True))
    unit_meta, unit_mean, unit_log, unit_fraction, unit_detected, unit_counts = unit_summary(obs, linear, log_rna, counts, ["compartment", STAGE, STUDY, "unit", "unit_kind"])
    col_meta, col_mean, col_log, col_fraction, _, _ = unit_summary(obs, linear, log_rna, counts, [STAGE, STATE, STUDY, "unit", "unit_kind", "compartment"])
    keep = col_meta.n_cells >= MIN_DISPLAY_CELLS
    col_meta, col_mean, col_log, col_fraction = col_meta.loc[keep].reset_index(drop=True), col_mean[keep], col_log[keep], col_fraction[keep]
    col_meta["stage_order"] = col_meta[STAGE].map({s: i for i, s in enumerate(STAGES)})
    col_meta["state_order"] = col_meta[STATE].map({s: i for i, s in enumerate(STATE_ORDER)})
    order = col_meta.sort_values(["stage_order", "state_order", STUDY, "unit"]).index.to_numpy()
    col_meta = col_meta.loc[order].reset_index(drop=True).drop(columns=["stage_order", "state_order"])
    col_meta.index = pd.Index([f"column_{i + 1:02d}" for i in range(len(col_meta))], name="column_id")
    summary = ad.AnnData(X=col_log[order].astype(np.float32), obs=col_meta, var=genes)
    summary.layers["mean_CP10k"] = col_mean[order].astype(np.float32)
    summary.layers["detection_fraction"] = col_fraction[order].astype(np.float32)
    # Population means retain observed state composition; state-level columns expose it.
    atlas_meta, atlas_mean, _, _, _, _ = unit_summary(obs, linear, log_rna, counts, [STAGE, STUDY, "unit"])
    summary.uns["atlas_stage_profile_CP10k"] = balance_by_stage(atlas_meta, atlas_mean, STAGES).astype(np.float32)
    summary.uns["selected_cell_ids"] = obs.index.to_numpy(dtype=str)
    summary.uns["input_sha256"] = hashlib.sha256(RNA_FILE.read_bytes()).hexdigest()
    summary.uns["input_file"] = str(RNA_FILE.relative_to(ROOT))
    summary.uns["known_genes_not_measured"] = np.array([g for g in KNOWN_GENES if g not in set(genes.gene_symbol)], dtype=str)
    summary.uns["cell_counts_by_stage"] = obs[STAGE].value_counts().reindex(STAGES).to_numpy()
    return summary, (unit_meta, unit_mean, unit_fraction, unit_detected, unit_counts)

## Differential expression

Require log2 fold change or exploratory log2 ratio above 0.5, 10% detection, at least three detected cells and support from two thirds of the evaluable units. Apply joint BH correction to all valid P values across the three formal contrasts, with FDR<0.05. Combine E16.5 apex/base counts within each biological replicate. Exploratory comparisons balance sources, samples and stages; libraries are not treated as independent biological replicates.


In [ ]:
#3. =========== Replicate-aware formal DE and exploratory stage markers
FORMAL_STUDY = "GSE195500"
FORMAL_STAGES = ["E14.5", "E16.5", "E18.5"]
LFC_THRESHOLD, FDR_THRESHOLD = 0.5, 0.05
EDGER_MIN_COUNT, EDGER_MIN_TOTAL_COUNT = 10, 15

R_SOURCE = r'''
suppressPackageStartupMessages(library(edgeR))
args <- commandArgs(trailingOnly=TRUE)
counts <- as.matrix(read.delim(args[1], row.names=1, check.names=FALSE))
meta <- read.delim(args[2], check.names=FALSE)
meta$stage <- factor(meta$stage, levels=c("E14.5", "E16.5", "E18.5"))
design <- model.matrix(~0+stage, meta)
y <- DGEList(counts=counts)
keep <- filterByExpr(y, design=design, min.count=as.numeric(args[4]), min.total.count=as.numeric(args[5]))
y <- calcNormFactors(y[keep,,keep.lib.sizes=FALSE], method="TMM")
fit <- glmQLFit(y, design, robust=TRUE)
out <- lapply(seq_len(ncol(design)), function(i) {
    contrast <- rep(-0.5, ncol(design)); contrast[i] <- 1
    tab <- topTags(glmTreat(fit, contrast=contrast, lfc=as.numeric(args[3])), n=Inf, sort.by="none")$table
    data.frame(gene_id=rownames(tab), stage=levels(meta$stage)[i], tab, row.names=NULL)
})
out <- do.call(rbind, out)
out$FDR <- p.adjust(out$PValue, method="BH")
write.table(out, args[6], sep="\t", row.names=FALSE, quote=FALSE)
samples <- data.frame(unit=colnames(y), y$samples, row.names=NULL)
write.table(samples, args[7], sep="\t", row.names=FALSE, quote=FALSE)
write.table(cpm(y), args[8], sep="\t", col.names=NA, quote=FALSE)
cat(sprintf("edgeR %s: %d tested genes, 9 biological replicates\n", packageVersion("edgeR"), nrow(y)))
cat(sprintf("Software: R=%s;edgeR=%s;limma=%s\n", getRversion(), packageVersion("edgeR"), packageVersion("limma")))
'''


def stage_results(summary, arrays):
    meta, means, fractions, detected, counts = arrays
    genes, eligible = summary.var, summary.var.eligible_rna.to_numpy()
    tables = []
    for scope in ["CVG", "SGN"]:
        take = meta.compartment == scope
        m, x, f, d = meta.loc[take].reset_index(drop=True), means[take], fractions[take], detected[take]
        stages = [s for s in STAGES if s in set(m[STAGE])]
        stage_mean, stage_fraction = balance_by_stage(m, x, stages), balance_by_stage(m, f, stages)
        background = eligible & ((f >= MIN_DETECTION) & (d >= MIN_DETECTED_CELLS)).any(axis=0)
        peaks = np.argmax(stage_mean, axis=0)
        for i, stage in enumerate(stages):
            target = m[STAGE] == stage
            rest = np.delete(stage_mean, i, axis=0).mean(axis=0)
            ratio = np.log2((stage_mean[i] + RATIO_PSEUDOCOUNT) / (rest + RATIO_PSEUDOCOUNT))
            agreement = (x[target] > rest) & (f[target] >= MIN_DETECTION) & (d[target] >= MIN_DETECTED_CELLS)
            support = agreement.sum(axis=0)
            source_support = pd.DataFrame(agreement.astype(float)).groupby(m.loc[target, STUDY].reset_index(drop=True)).mean()
            balanced_support = source_support.mean(axis=0).to_numpy()
            minimum_source_support = source_support.min(axis=0).to_numpy()
            n_units = int(target.sum())
            qualified = background & (ratio > LFC_THRESHOLD) & (stage_fraction[i] >= MIN_DETECTION) & (d[target].sum(axis=0) >= MIN_DETECTED_CELLS) & (balanced_support >= MIN_UNIT_SUPPORT) & (minimum_source_support > 0)
            tables.append(pd.DataFrame({"gene_id": genes.index, "stage": stage, "compartment": scope, "evidence_class": "exploratory", "tested": background, "qualified": qualified, "effect": ratio, "effect_definition": "balanced_CP10k_log2_ratio", "target_mean_CP10k": stage_mean[i], "reference_mean_CP10k": rest, "target_detection_fraction": stage_fraction[i], "support_units": support, "source_balanced_support_fraction": balanced_support, "minimum_source_support_fraction": minimum_source_support, "target_units": n_units, "single_library": n_units == 1, "peak_in_scope": peaks == i, "PValue": np.nan, "FDR": np.nan}))
    take = (meta[STUDY] == FORMAL_STUDY) & (meta.compartment == "SGN")
    pb = meta.loc[take].copy().reset_index(drop=True)
    assert pb.unit.is_unique and pb.groupby(STAGE).size().reindex(FORMAL_STAGES).eq(3).all() and pb.n_cells.min() >= MIN_FORMAL_CELLS
    pb.index = pd.Index(pb.unit, name="replicate_id")
    raw = counts[take]
    with tempfile.TemporaryDirectory(prefix="edgeR-", dir=TMP_DIR) as td:
        tmp = Path(td)
        # Uses counts.tsv generated by the edgeR input preparation step.
        pd.DataFrame(raw[:, eligible].T, index=genes.index[eligible], columns=pb.unit).to_csv(ROOT, sep="\t")
        # Uses samples.tsv generated by the edgeR input preparation step.
        pb.rename(columns={STAGE: "stage"})[["stage", "unit", "n_cells"]].to_csv(ROOT, sep="\t", index=False)
        # Set the file or directory path.
        (ROOT).write_text(R_SOURCE, encoding="utf-8")
        env = dict(os.environ, LC_ALL="C", R_USER=str(TMP_DIR))
        # Set the file or directory path. Uses counts.tsv generated by the edgeR input preparation step. Uses samples.tsv generated by the edgeR input preparation step. Uses results.tsv generated by the edgeR fitting step. Uses normalization.tsv generated by the edgeR fitting step. Uses CPM.tsv generated by the edgeR fitting step.
        run = subprocess.run([str(RSCRIPT), "--vanilla", str(ROOT), str(ROOT), str(ROOT), str(LFC_THRESHOLD), str(EDGER_MIN_COUNT), str(EDGER_MIN_TOTAL_COUNT), str(ROOT), str(ROOT), str(ROOT)], env=env, capture_output=True, text=True, check=True)
        messages = run.stdout.strip().splitlines()
        print(messages[0])
        summary.uns["software"] = dict(item.split("=") for item in messages[-1].removeprefix("Software: ").split(";"))
        summary.uns["software"].update(numpy=np.__version__, pandas=pd.__version__, matplotlib=matplotlib.__version__)
        if run.stderr.strip():
            print(run.stderr.strip())
        # Uses results.tsv generated by the edgeR fitting step.
        formal = pd.read_csv(ROOT, sep="\t")
        # Uses normalization.tsv generated by the edgeR fitting step.
        normalization = pd.read_csv(ROOT, sep="\t").set_index("unit")
        # Uses CPM.tsv generated by the edgeR fitting step.
        cpm = pd.read_csv(ROOT, sep="\t", index_col=0).reindex(columns=pb.unit)
    formal.rename(columns={"logFC": "effect"}, inplace=True)
    tested_ids = pd.Index(cpm.index, name="gene_id")
    gene_idx = genes.index.get_indexer(tested_ids)
    formal_mean = np.array([cpm.loc[:, pb[STAGE] == s].mean(axis=1).to_numpy() for s in FORMAL_STAGES])
    for i, stage in enumerate(FORMAL_STAGES):
        rows = formal.loc[formal.stage == stage].set_index("gene_id").reindex(tested_ids).copy()
        target = pb[STAGE] == stage
        reference = np.exp(np.log(np.delete(formal_mean, i, axis=0) + RATIO_PSEUDOCOUNT).mean(axis=0)) - RATIO_PSEUDOCOUNT
        f, d = fractions[take][:, gene_idx], detected[take][:, gene_idx]
        support = ((cpm.loc[:, target].T.to_numpy() > reference) & (f[target] >= MIN_DETECTION) & (d[target] >= MIN_DETECTED_CELLS)).sum(axis=0)
        rows["compartment"], rows["evidence_class"], rows["tested"] = "SGN", "formal", True
        rows["effect_definition"] = "edgeR_target_vs_geometric_mean_other_two"
        rows["target_mean_CP10k"] = means[take][target][:, gene_idx].mean(axis=0)
        rows["reference_mean_CP10k"] = np.nan
        rows["target_detection_fraction"] = f[target].mean(axis=0)
        rows["support_units"], rows["target_units"], rows["single_library"] = support, 3, False
        rows["source_balanced_support_fraction"], rows["minimum_source_support_fraction"] = support / 3, support / 3
        rows["peak_in_scope"] = np.argmax(formal_mean, axis=0) == i
        rows["qualified"] = (rows.effect > LFC_THRESHOLD) & (rows.FDR < FDR_THRESHOLD) & (rows.target_detection_fraction >= MIN_DETECTION) & (support >= 2)
        tables.append(rows.reset_index())
    summary.uns["formal_counts"] = raw.astype(np.int64)
    summary.uns["formal_replicates"] = pb
    summary.uns["formal_normalization"] = normalization
    summary.uns["formal_tested_gene_ids"] = tested_ids.to_numpy(dtype=str)
    summary.var["formal_tested"] = summary.var_names.isin(tested_ids)
    summary.uns["formal_normalized_CPM"] = cpm.T.to_numpy(dtype=np.float32)
    for table in tables:
        table["logCPM"] = table.get("logCPM", np.nan)
        table["reference_mean_CP10k"] = table.reference_mean_CP10k.astype(float)
    results = pd.concat(tables, ignore_index=True).merge(genes[["gene_symbol", "mgi_id"]], left_on="gene_id", right_index=True, how="left")
    return results

## Select display genes

Keep the selected 198 rows, with at most 22 per stage, while saving the complete selection results separately. Label three genes per stage; asterisks indicate candidate functions rather than stage specificity established by literature. Plotting Z scores balance stages, sources and samples, with state-cell-count weights within each unit. Clip only the displayed colors at +/-3.5.


In [ ]:
#4. =========== Representative heatmap rows and two key data files
MAX_ROWS_PER_STAGE = 22              # Display cap; full qualified lists remain in the DEG table.
MAX_KNOWN_ROWS_PER_STAGE = 3          # Prioritize measured, qualified literature candidates.
Z_LIMIT = 3.5                        # Common display limit; saved Z-scores remain unclipped.
HEATMAP_NODES = [(-3.5, "#8EA4CE"), (-1.5, "#C5CEE2"), (0, "#F5F0E3"), (1, "#F4DE9D"), (2, "#EDA264"), (3.5, "#C84A44")]
LABEL_SPECS = [
    ("E9.5", "Sox2", False, "Inner-ear neurogenesis: genetic function", "Neurogenic competence and CVG specification", "https://pmc.ncbi.nlm.nih.gov/articles/PMC5481345/"),
    ("E9.5", "Lfng", True, "Otic progenitor-domain expression; CVG-role candidate", "Notch modulation; early otic progenitor program", "https://pubmed.ncbi.nlm.nih.gov/9547240/"),
    ("E9.5", "Hmga2", True, "General neural-progenitor function; CVG-role candidate", "Chromatin regulation and fetal neural self-renewal", "https://pmc.ncbi.nlm.nih.gov/articles/PMC2582221/"),
    ("E10.5", "Neurod1", False, "Inner-ear neurogenesis: genetic function", "Neuroblast differentiation and survival", "https://pubmed.ncbi.nlm.nih.gov/35252209/"),
    ("E10.5", "Tubb3", False, "Developing CVG expression marker", "Neuronal microtubule and differentiation program", "https://pmc.ncbi.nlm.nih.gov/articles/PMC10592509/"),
    ("E10.5", "Dll3", True, "General neural differentiation; CVG-role candidate", "Delta-family neurogenic differentiation program", "https://pmc.ncbi.nlm.nih.gov/articles/PMC2698949/"),
    ("E11.5", "Insm1", False, "Otic neurogenesis: genetic function", "Proliferation of delaminated otic neuronal progenitors", "https://pubmed.ncbi.nlm.nih.gov/26545349/"),
    ("E11.5", "Meis2", True, "Inner-ear morphogenesis; neuronal-role candidate", "Transcriptional control of otic development", "https://pmc.ncbi.nlm.nih.gov/articles/PMC8194062/"),
    ("E11.5", "Igf2bp1", True, "General neuronal function; auditory-role candidate", "RNA localization and axonal local translation", "https://pmc.ncbi.nlm.nih.gov/articles/PMC4556669/"),
    ("E13.5", "Shox2", False, "Otic neuronal expression and lineage evidence", "Transcription factor marking otic neuronal differentiation", "https://www.sciencedirect.com/science/article/pii/S2211124722002832"),
    ("E13.5", "Nrp2", False, "Auditory axon guidance: functional evidence", "Semaphorin coreceptor in neuronal targeting", "https://pmc.ncbi.nlm.nih.gov/articles/PMC4566076/"),
    ("E13.5", "Bcl11a", True, "General sensory-circuit function; auditory-role candidate", "Neuronal maturation and sensory circuitry", "https://pmc.ncbi.nlm.nih.gov/articles/PMC4067532/"),
    ("E14.5", "Myt1", False, "Developing CVG expression marker", "Neuronal transcriptional differentiation program", "https://www.informatics.jax.org/assay/MGI:7493750"),
    ("E14.5", "Six1", False, "Inner-ear neuronal fate: functional evidence", "Eya1-Six1 neuronal specification and differentiation", "https://pubmed.ncbi.nlm.nih.gov/22513373/"),
    ("E14.5", "Shh", False, "SGN-source signaling: genetic function", "SGN signal regulating cochlear epithelial differentiation", "https://pubmed.ncbi.nlm.nih.gov/23918393/"),
    ("E15.5", "Gata3", False, "SGN wiring and differentiation: genetic function", "Neurite growth, pathfinding and differentiation timing", "https://pubmed.ncbi.nlm.nih.gov/23426694/"),
    ("E15.5", "Eya1", False, "Inner-ear neuronal fate: functional evidence", "Eya1-Six1 neuronal specification and differentiation", "https://pubmed.ncbi.nlm.nih.gov/22513373/"),
    ("E15.5", "Etv4", True, "Published embryonic SGN regulator candidate", "ETS factor associated with the first SGN branch", "https://pubmed.ncbi.nlm.nih.gov/36409884/"),
    ("E16.5", "Mafb", False, "Auditory postsynaptic differentiation: genetic function", "Postsynaptic specialization at auditory ribbon synapses", "https://elifesciences.org/articles/01341"),
    ("E16.5", "Casz1", True, "Otic neuronal expression; regulatory-role candidate", "Zinc-finger transcriptional developmental program", "https://www.informatics.jax.org/marker/MGI:1196251"),
    ("E16.5", "Id1", True, "SGN branch expression; regulatory-role candidate", "bHLH-related differentiation-state regulation", "https://pubmed.ncbi.nlm.nih.gov/36409884/"),
    ("E17.5", "Th", False, "SGN subtype expression marker", "Catecholamine synthesis program; temporal association only", "https://pubmed.ncbi.nlm.nih.gov/36409884/"),
    ("E17.5", "Adcy1", True, "General neural circuit function; auditory-role candidate", "Calcium/calmodulin-responsive cAMP signaling", "https://pubmed.ncbi.nlm.nih.gov/9662407/"),
    ("E17.5", "Plxna1", True, "General axon guidance; auditory-role candidate", "Plexin-mediated axonal guidance signaling", "https://www.nature.com/articles/nn.3893"),
    ("E18.5", "Lypd1", False, "Embryonic SGN subtype expression marker", "Cell-surface program associated with Type Ic neurons", "https://pubmed.ncbi.nlm.nih.gov/36409884/"),
    ("E18.5", "Calb2", False, "Embryonic SGN subtype expression marker", "Calcium-binding neuronal subtype program", "https://pubmed.ncbi.nlm.nih.gov/36409884/"),
    ("E18.5", "Nefh", False, "Established neuronal cytoskeleton marker", "Neurofilament-heavy axonal maturation program", "https://www.informatics.jax.org/marker/MGI:97309"),
]


def update_plot_data(summary):
    rows, cols = summary.uns["display_rows"], summary.obs
    unit_cells = cols.groupby([STAGE, STUDY, "unit"], observed=True).n_cells.transform("sum")
    units = cols.groupby([STAGE, STUDY], observed=True).unit.transform("nunique")
    sources = cols.groupby(STAGE, observed=True)[STUDY].transform("nunique")
    weights = (cols.n_cells / unit_cells / units / sources / len(STAGES)).to_numpy(dtype=float)
    assert np.isclose(weights.sum(), 1)
    values = summary.X[:, summary.var_names.get_indexer(rows.gene_id)].T.astype(np.float64)
    row_mean = values @ weights
    row_sd = np.sqrt(((values - row_mean[:, None]) ** 2) @ weights)
    z = (values - row_mean[:, None]) / row_sd[:, None]
    assert np.isfinite(z).all()
    labels = pd.DataFrame(LABEL_SPECS, columns=["stage", "gene_symbol", "candidate", "literature_evidence", "function", "reference"])
    fields = ["stage", "gene_symbol", "gene_id", "compartment", "evidence_class", "effect", "FDR", "target_detection_fraction", "support_units", "target_units"]
    labels = labels.merge(rows[fields], on=["stage", "gene_symbol"], validate="one_to_one")
    assert len(labels) == 27 and labels.groupby("stage").size().eq(3).all()
    rows["labelled"] = rows.gene_symbol.isin(labels.gene_symbol)
    rows["label_is_candidate"] = rows.gene_symbol.isin(labels.loc[labels.candidate, "gene_symbol"])
    summary.obs["standardization_weight"] = weights
    summary.uns["heatmap_zscore"] = z.astype(np.float32)
    summary.uns["row_mean_log1p_CP10k"], summary.uns["row_sd_log1p_CP10k"] = row_mean, row_sd
    summary.uns["marker_labels"] = labels
    summary.uns["plot_settings"] = {"z_limit": Z_LIMIT, "color_values": np.array([x[0] for x in HEATMAP_NODES]), "colors": np.array([x[1] for x in HEATMAP_NODES]), "labels_per_stage": 3}
    summary.uns["methods"]["expression"] = "Original RNA X=log1p(CP10k), without renormalization. Columns are unit x existing cell_type_plot mean X. Weighted row Z: stages, sources within stage, and units within source equally weighted; original states weighted by displayed cell counts within unit. Common color limit +/-3.5; stored Z is unclipped."
    summary.uns["methods"]["marker_labels"] = "Three qualified, measured, stage-peak genes per period, already in the existing displayed set. Published marker/function evidence is distinguished from candidate auditory developmental roles by asterisk. Literature does not establish temporal specificity here; E9.5/E17.5 remain single-library exploratory associations. Espn is retained as a data-supported row, without an E9.5 known-neuronal-marker claim."
    return summary


def save_results(summary, results):
    peak_stage = pd.Series(np.array(STAGES)[np.argmax(summary.uns["atlas_stage_profile_CP10k"], axis=0)], index=summary.var_names)
    pool = results.loc[results.qualified & results.peak_in_scope].copy()
    pool = pool.loc[pool.stage == pool.gene_id.map(peak_stage)].copy()
    pool["formal_priority"] = pool.evidence_class.eq("formal").astype(int)
    pool = pool.sort_values(["formal_priority", "FDR", "effect", "gene_id"], ascending=[False, True, False, True], na_position="last").drop_duplicates("gene_id")
    selected = []
    for stage in STAGES:
        candidates = pool.loc[pool.stage == stage]
        known = candidates.loc[candidates.gene_symbol.isin(KNOWN_GENES)].head(MAX_KNOWN_ROWS_PER_STAGE)
        chosen = pd.concat([known, candidates]).drop_duplicates("gene_id").head(MAX_ROWS_PER_STAGE)
        # Order expression profiles within each stage block; break ties by gene ID.
        if len(chosen):
            profile = summary.uns["atlas_stage_profile_CP10k"][:, summary.var_names.get_indexer(chosen.gene_id)]
            centroid = (np.arange(len(STAGES))[:, None] * profile).sum(axis=0) / profile.sum(axis=0)
            chosen = chosen.assign(expression_centroid=centroid).sort_values(["expression_centroid", "gene_id"])
            selected.append(chosen)
    display = pd.concat(selected, ignore_index=True).drop(columns="formal_priority")
    display.index = pd.Index([f"row_{i + 1:03d}" for i in range(len(display))], name="row_id")
    assert not display.gene_id.duplicated().any()
    keys = ["gene_id", "stage", "compartment", "evidence_class"]
    results["gene_in_heatmap"] = results.gene_id.isin(display.gene_id)
    results["displayed"] = pd.MultiIndex.from_frame(results[keys]).isin(pd.MultiIndex.from_frame(display[keys]))
    summary.uns["display_rows"] = display
    summary.uns["stage_order"], summary.uns["stage_colors"], summary.uns["study_colors"] = np.array(STAGES), STAGE_COLORS, STUDY_COLORS
    summary.uns["parameters"] = {"min_detection": MIN_DETECTION, "min_detected_cells": MIN_DETECTED_CELLS, "min_unit_support": MIN_UNIT_SUPPORT, "ratio_pseudocount_CP10k": RATIO_PSEUDOCOUNT, "min_display_cells": MIN_DISPLAY_CELLS, "min_formal_cells": MIN_FORMAL_CELLS, "log2FC_threshold": LFC_THRESHOLD, "joint_FDR_threshold": FDR_THRESHOLD, "max_display_rows_per_stage": MAX_ROWS_PER_STAGE, "edgeR_min_count": EDGER_MIN_COUNT, "edgeR_min_total_count": EDGER_MIN_TOTAL_COUNT, "seed": SEED}
    summary.uns["methods"] = {"formal_DE": "GSE195500 SGN; E16 Apex/Base summed within biological replicate; ~0+stage; TMM; robust edgeR QL; glmTreat lfc=0.5; each stage vs mean log-expression of the other two; BH across all three contrasts; positive effects; target detection>=0.1 and >=2/3 supporting replicates against the observed TMM-CPM geometric reference.", "exploratory_markers": "Separate CVG/SGN. Within source: units equally weighted; sources equally weighted within stage; other stages equally weighted. No P-values. Ratio uses mean CP10k+0.1; support fraction >=2/3 after equal-source averaging, with support in every available source; detection>=0.1 and >=3 detected cells per supporting unit. Plates are observational units, not verified biological replicates.", "composition": "Population-stage contrasts include original-state proportion changes. Heatmap columns separate original states. State means do not establish within-state causal age effects; stage/source/technology confounding remains for exploratory comparisons.", "expression": "Original RNA X=log1p(CP10k), without renormalization; heatmap columns: mean X within unit x existing cell_type_plot; ordinary row z-score across the displayed columns (population SD).", "display_selection": "Qualified positive genes; peak in comparison scope and in balanced atlas CP10k; one appearance per gene. Stage blocks sorted by expression centroid, up to 22 rows per stage; up to 3 known qualified genes prioritized. Display selection is separate from full qualified lists.", "unmeasured": "Only common measured 11467-gene axis is analyzed. Absent genes are unmeasured, not zero. Endogenous Ercc genes are retained; only ERCC- spike-ins and mitochondrial genes are excluded.", "GO_background": "Formal: formal_tested_gene_ids. Exploratory: tested=True genes within the same compartment. GO not run in this stage; use complete qualified lists or justified modules, not displayed rows alone.", "data_contents": "X=unit-state mean log1p(CP10k); layers=mean CP10k and detection fraction; uns=formal raw pseudobulks, tested genes, normalization, complete plotting matrix, row/column metadata and parameters; complete formal/exploratory results in companion Stage-DEG.tsv.gz."}
    summary = update_plot_data(summary)
    # Object strings keep the H5AD readable in older AnnData versions.
    for frame in [summary.obs, summary.var, display, summary.uns["formal_replicates"], summary.uns["marker_labels"]]:
        for column in frame.select_dtypes(include="string").columns:
            frame[column] = frame[column].astype(object)
    for directory in [DATA_DIR, FIG_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    # Save the plotted summaries and keep raw counts in the input file.
    profile = summary.uns["atlas_stage_profile_CP10k"][:, summary.var_names.get_indexer(display.gene_id)]
    summary = summary[:, display.gene_id].copy()
    summary.uns["atlas_stage_profile_CP10k"] = profile
    for key in ["formal_counts", "formal_normalized_CPM", "selected_cell_ids"]:
        summary.uns.pop(key)
    summary.uns["measured_gene_count"] = 11467
    summary.uns["data_contents"] = "Plotted RNA mean expression, detection fractions, exact Z-scores, row/column metadata, parameters, software versions, input checksum and formal tested-gene IDs. Complete DE results are in Stage-DEG.tsv.gz; raw counts remain in the unchanged original RNA input."
    summary.uns["methods"]["data_contents"] = summary.uns["data_contents"]
    summary.write_h5ad(SUMMARY_FILE, compression="gzip")
    results.to_csv(DEG_FILE, sep="\t", index=False, compression="gzip", float_format="%.10g")
    print("Formal upregulated DEGs:", results.loc[results.qualified & results.evidence_class.eq("formal")].groupby("stage").size().to_dict())
    print("Displayed rows:", display.groupby("stage", sort=False).size().to_dict())
    return summary

## Reuse saved results

Save `Stage-heatmap.h5ad` and `Stage-DEG.tsv.gz`. If the cache is absent, rebuild from the integration object; otherwise reuse it. Set `REBUILD_RESULTS=True` to rerun differential expression. Appearance changes only require reloading the summaries and rerunning the plotting and export sections.


In [ ]:
#5. =========== Load or rebuild the two key results
if REBUILD_RESULTS or not SUMMARY_FILE.exists():
    summary, arrays = build_summaries()
    results = stage_results(summary, arrays)
    summary = save_results(summary, results)
else:
    assert DEG_FILE.is_file(), "The complete DEG table is missing."
    summary = ad.read_h5ad(SUMMARY_FILE)
    summary = update_plot_data(summary)
    summary.write_h5ad(SUMMARY_FILE, compression="gzip")
assert summary.uns["heatmap_zscore"].shape == (198, 89)
assert summary.uns["marker_labels"].groupby("stage").size().eq(3).all()
print("Heatmap data:", summary.shape, "| 27 evidence-annotated labels")

## Heatmap layout

Keep the selected color scale, row and column order, and gene labels. Use Arial and export PNGs at 600 dpi.


In [ ]:
#6. =========== Heatmap layout
FIGSIZE = (7.2, 7.4)                  # Inches; PDF retains vector text and cell rectangles.
PNG_DPI = 600
LABEL_FONTSIZE, LABEL_GAP_PT = 7.0, 9.5


def make_heatmap_layout(summary):
    plt.rcParams.update({"font.family": "Arial", "font.size": 7.5, "pdf.fonttype": 42, "axes.linewidth": 0.5})
    rows, cols = summary.uns["display_rows"], summary.obs
    z = summary.uns["heatmap_zscore"]
    n_rows = len(rows)
    edges = np.concatenate([i + np.linspace(0, 1, sum(cols[STAGE].astype(str) == s) + 1)[:-1] for i, s in enumerate(STAGES)] + [np.array([len(STAGES)])])
    cmap = LinearSegmentedColormap.from_list("C_panel", [((value + Z_LIMIT) / (2 * Z_LIMIT), color) for value, color in HEATMAP_NODES])
    fig = plt.figure(figsize=FIGSIZE)
    left, bottom, width, height = 0.13, 0.155, 0.65, 0.74
    ax = fig.add_axes([left, bottom, width, height])
    ax.pcolormesh(edges, np.arange(n_rows + 1), z, cmap=cmap, vmin=-Z_LIMIT, vmax=Z_LIMIT, shading="flat", linewidth=0, rasterized=False)
    ax.set(xlim=(0, len(STAGES)), ylim=(n_rows, 0), xticks=[], yticks=[])
    for spine in ax.spines.values():
        spine.set_visible(False)
    for x in range(1, len(STAGES)):
        ax.axvline(x, color="white", lw=0.7)
    row_stage = rows.stage.to_numpy(dtype=str)
    for stage in STAGES:
        ids = np.where(row_stage == stage)[0]
        if len(ids):
            a, b = ids.min(), ids.max() + 1
            ax.axhline(b, color="white", lw=0.7)
            ax.text(-0.27, (a + b) / 2, stage, ha="right", va="center", fontsize=8, clip_on=False)
            ax.plot([-0.13, -0.13], [a, b], color=STAGE_COLORS[stage], lw=3, solid_capstyle="butt", clip_on=False)
    # Narrow row stripe distinguishes formal DE from exploratory markers.
    evidence_ax = fig.add_axes([left - 0.006, bottom, 0.004, height])
    ev = rows.evidence_class.eq("formal").to_numpy()
    evidence_ax.pcolormesh([0, 1], np.arange(n_rows + 1), ev.astype(int)[:, None], cmap=matplotlib.colors.ListedColormap(["#C8C8C8", "#333333"]), vmin=0, vmax=1, shading="flat")
    evidence_ax.set_ylim(n_rows, 0)
    evidence_ax.axis("off")
    strip_labels = ["Stage", "Source", "Scope"]
    strip_colors = [[STAGE_COLORS[s] for s in cols[STAGE].astype(str)], [STUDY_COLORS[s] for s in cols[STUDY].astype(str)], ["#DABCA0" if s == "CVG" else "#AFC2BC" for s in cols.compartment.astype(str)]]
    for j, (label, colors) in enumerate(zip(strip_labels, strip_colors)):
        strip = fig.add_axes([left, bottom + height + 0.004 + (2 - j) * 0.009, width, 0.006])
        strip.pcolormesh(edges, [0, 1], np.array([np.arange(len(cols))]), cmap=matplotlib.colors.ListedColormap(colors), vmin=-0.5, vmax=len(cols) - 0.5, shading="flat")
        strip.axis("off")
        strip.text(-0.02, 0.5, label, transform=strip.transAxes, ha="right", va="center", fontsize=6.2)
    for i, stage in enumerate(STAGES):
        ax.text(i + 0.5, -n_rows * 0.048, stage, ha="center", va="bottom", fontsize=7.3, clip_on=False)
    label_artists = []
    min_gap = LABEL_GAP_PT * n_rows / (height * FIGSIZE[1] * 72)
    for stage in STAGES:
        ids = np.where((rows.stage == stage) & rows.labelled)[0]
        bounds = np.where(rows.stage == stage)[0]
        positions = ids.astype(float) + 0.5
        for i in range(1, len(positions)):
            positions[i] = max(positions[i], positions[i - 1] + min_gap)
        lower, upper = bounds[0] + min_gap / 2, bounds[-1] + 1 - min_gap / 2
        if positions[-1] > upper:
            positions[-1] = upper
            for i in range(len(positions) - 2, -1, -1):
                positions[i] = min(positions[i], positions[i + 1] - min_gap)
        if positions[0] < lower:
            positions += lower - positions[0]
        for rid, pos in zip(ids, positions):
            ax.plot([len(STAGES), len(STAGES) + 0.12, len(STAGES) + 0.20], [rid + 0.5, rid + 0.5, pos], color="0.40", lw=0.45, clip_on=False)
            name = rows.iloc[rid].gene_symbol + ("*" if rows.iloc[rid].label_is_candidate else "")
            label_artists.append(ax.text(len(STAGES) + 0.25, pos, name, ha="left", va="center", fontsize=LABEL_FONTSIZE, fontstyle="italic", clip_on=False))
    fig.text(left, 0.984, "Embryonic stage-associated RNA expression", fontsize=10, fontweight="bold", va="top")
    fig.text(left, 0.958, f"{n_rows} genes | {len(cols)} sample-state means | 6,995 cells used for gene selection", fontsize=7.2, va="top", color="0.3")
    cax = fig.add_axes([0.91, 0.80, 0.017, 0.10])
    cb = fig.colorbar(matplotlib.cm.ScalarMappable(norm=matplotlib.colors.Normalize(-Z_LIMIT, Z_LIMIT), cmap=cmap), cax=cax, ticks=[-3.5, -2, 0, 1, 2, 3.5])
    cb.ax.set_title("Row Z", fontsize=7, pad=4)
    cb.ax.tick_params(labelsize=6.5, width=0.4, length=2, pad=2)
    cb.outline.set_visible(False)
    handles = [Patch(facecolor=STUDY_COLORS[s], label=s) for s in STUDY_COLORS]
    fig.legend(handles=handles, loc="lower left", bbox_to_anchor=(left - 0.01, 0.083), ncol=4, frameon=False, fontsize=6.8, handlelength=1, handletextpad=0.4, columnspacing=1.5)
    fig.legend(handles=[Patch(facecolor="#DABCA0", label="CVG"), Patch(facecolor="#AFC2BC", label="SGN"), Patch(facecolor="#333333", label="Formal DEG"), Patch(facecolor="#C8C8C8", label="Exploratory marker")], loc="lower left", bbox_to_anchor=(left - 0.01, 0.060), ncol=4, frameon=False, fontsize=6.8, handlelength=1, handletextpad=0.4, columnspacing=1.4)
    fig.text(left, 0.050, "Columns: sample/library x original state; stage panels have equal width.", fontsize=6.7, va="top")
    fig.text(left, 0.033, "Stage/source/sample-balanced row Z, colors clipped at +/-3.5; stage contrasts retain state composition.", fontsize=6.2, va="top")
    fig.text(left, 0.016, "* Candidate role in this developmental program; literature evidence does not establish stage specificity.", fontsize=6.2, va="top")
    fig.canvas.draw()
    boxes = [artist.get_window_extent(fig.canvas.get_renderer()) for artist in label_artists]
    assert len(boxes) == 27 and all(not a.overlaps(b) for a, b in zip(boxes, boxes[1:]))
    return fig

## Editable vector export

Initialize Illustrator before export. Create four native layers, `Axes`, `Text`, `Legend` and `Graphics`, each with one page clipping mask. Draw heatmap cells as vector rectangles, lines as paths and labels as complete text frames. Keep legend text and shapes together. Retain Illustrator editing data in the PDF and reopen the file for verification.


In [ ]:
#7. =========== Vector scene and four native Illustrator layers
def vector_scene(fig):
    fig.canvas.draw()
    renderer, scale = fig.canvas.get_renderer(), 72 / fig.dpi
    scene = {"size": [float(v * 72) for v in fig.get_size_inches()], "paths": [], "texts": []}

    def path(vertices, layer, fill=None, stroke=None, width=0, closed=False):
        pts = [[round(float(x) * scale, 4), round(float(y) * scale, 4)] for x, y in vertices]
        if closed and pts[0] == pts[-1]:
            pts = pts[:-1]
        scene["paths"].append({"p": pts, "layer": layer, "fill": fill, "stroke": stroke, "width": float(width), "closed": closed})

    def rgb(color):
        return [round(float(v) * 255, 4) for v in to_rgba(color)[:3]]

    def text(artist, layer):
        if not artist.get_visible() or not artist.get_text():
            return
        _, lines, _ = artist._get_layout(renderer)
        x, y = artist.get_transform().transform(artist.get_unitless_position())
        font = artist.get_fontproperties()
        face = "Arial-ItalicMT" if font.get_style() == "italic" else ("Arial-BoldMT" if font.get_weight() in ["bold", 700] else "ArialMT")
        for content, _, ox, oy in lines:
            scene["texts"].append({"text": content, "x": round(float(x + ox) * scale, 4), "y": round(float(y + oy) * scale, 4), "size": float(font.get_size_in_points()), "font": face, "color": rgb(artist.get_color()), "layer": layer})

    for i, ax in enumerate(fig.axes):
        layer = "Legend" if i == len(fig.axes) - 1 else "Graphics"
        for mesh in ax.collections:
            if isinstance(mesh, QuadMesh):
                for shape, color in zip(mesh.get_paths(), mesh.get_facecolors()):
                    path(mesh.get_transform().transform(shape.vertices), layer, fill=rgb(color), closed=True)
        for line in ax.lines:
            if line.get_visible() and line.get_linestyle() not in ["None", "", " "]:
                name = "Axes" if np.allclose(to_rgba(line.get_color())[:3], [1, 1, 1]) else "Graphics"
                path(line.get_transform().transform(line.get_path().vertices), name, stroke=rgb(line.get_color()), width=line.get_linewidth())
        for artist in ax.texts:
            text(artist, "Text")
        text(ax.title, "Legend" if layer == "Legend" else "Text")
        if layer == "Legend":
            for tick in ax.yaxis.get_major_ticks():
                text(tick.label2, "Legend")
                line = tick.tick2line
                if line.get_visible():
                    point = line.get_transform().transform(line.get_path().vertices[0])
                    path([point, point + [line.get_markersize() / scale, 0]], "Axes", stroke=rgb(line.get_color()), width=line.get_markeredgewidth())
    for legend in fig.legends:
        for handle in legend.legend_handles:
            path(handle.get_transform().transform(handle.get_path().vertices), "Legend", fill=rgb(handle.get_facecolor()), closed=True)
        for artist in legend.get_texts():
            text(artist, "Legend")
    for artist in fig.texts:
        text(artist, "Text")
    return scene


ILLUSTRATOR_JS = r"""
(function () {
    var oldLevel = app.userInteractionLevel;
    app.userInteractionLevel = UserInteractionLevel.DONTDISPLAYALERTS;
    function rgb(values) {var c=new RGBColor();c.red=values[0];c.green=values[1];c.blue=values[2];return c;}
    function json(v) {
        if (v === null) return "null";
        if (typeof v === "number" || typeof v === "boolean") return String(v);
        if (typeof v === "string") return '"'+v.replace(/\\/g,'\\\\').replace(/"/g,'\\"').replace(/\r/g,'\\r').replace(/\n/g,'\\n')+'"';
        var parts=[];if(v instanceof Array){for(var i=0;i<v.length;i++)parts.push(json(v[i]));return '['+parts.join(',')+']';}
        for(var k in v)if(v.hasOwnProperty(k))parts.push(json(k)+':'+json(v[k]));return '{'+parts.join(',')+'}';
    }
    var doc = app.documents.add(DocumentColorSpace.RGB, S.size[0], S.size[1]);
    try {
        var groups={}, masks={}, names=["Graphics", "Legend", "Text", "Axes"];
        for(var i=0;i<names.length;i++) {
            var layer=i===0?doc.layers[0]:doc.layers.add();layer.name=names[i];
            var group=layer.groupItems.add();group.name=names[i];groups[names[i]]=group;
            var mask=group.pathItems.rectangle(S.size[1],0,S.size[0],S.size[1]);
            mask.name="Page mask";mask.filled=false;mask.stroked=false;mask.clipping=true;masks[names[i]]=mask;
        }
        for(var i=0;i<S.paths.length;i++) {
            var r=S.paths[i], p=groups[r.layer].pathItems.add();p.setEntirePath(r.p);p.closed=r.closed;
            p.filled=r.fill!==null;if(p.filled)p.fillColor=rgb(r.fill);
            p.stroked=r.stroke!==null;if(p.stroked){p.strokeColor=rgb(r.stroke);p.strokeWidth=r.width;}
        }
        for(var i=0;i<S.texts.length;i++) {
            var r=S.texts[i],t=groups[r.layer].textFrames.pointText([r.x,r.y]);t.contents=r.text;
            t.textRange.characterAttributes.textFont=app.textFonts.getByName(r.font);
            t.textRange.characterAttributes.size=r.size;t.textRange.characterAttributes.fillColor=rgb(r.color);
        }
        for(var i=0;i<names.length;i++){masks[names[i]].zOrder(ZOrderMethod.BRINGTOFRONT);groups[names[i]].clipped=true;}
        var opt=new PDFSaveOptions();opt.compatibility=PDFCompatibility.ACROBAT6;
        opt.preserveEditability=true;opt.acrobatLayers=true;opt.generateThumbnails=false;opt.viewAfterSaving=false;
        doc.saveAs(new File(S.pdf),opt);
        var png=new ExportOptionsPNG24();png.horizontalScale=S.dpi/72*100;png.verticalScale=S.dpi/72*100;
        png.antiAliasing=true;png.artBoardClipping=true;png.transparency=false;
        doc.exportFile(new File(S.png.replace(/\.png$/i,"")),ExportType.PNG24,png);
        doc.close(SaveOptions.DONOTSAVECHANGES);doc=null;
        var check=app.open(new File(S.pdf));
        var report={version:app.version,layers:[],clipping_groups:0,clipping_paths:0,raster_items:check.rasterItems.length,placed_items:check.placedItems.length,texts:[],fonts:[],paths:check.pathItems.length};
        for(var i=0;i<check.layers.length;i++){
            var layer=check.layers[i],item={name:layer.name,groups:layer.groupItems.length,masks:0};
            for(var j=0;j<layer.groupItems.length;j++)if(layer.groupItems[j].clipped){report.clipping_groups++;item.masks++;}
            report.layers.push(item);
        }
        for(var i=0;i<check.pathItems.length;i++)if(check.pathItems[i].clipping)report.clipping_paths++;
        var seen={};for(var i=0;i<check.textFrames.length;i++){
            var t=check.textFrames[i];report.texts.push(t.contents);var f=t.textRange.characterAttributes.textFont.name;
            if(!seen[f]){seen[f]=true;report.fonts.push(f);}
        }
        check.close(SaveOptions.DONOTSAVECHANGES);
        return json(report);
    } finally {if(doc)doc.close(SaveOptions.DONOTSAVECHANGES);app.userInteractionLevel=oldLevel;}
})()
"""


def export_editable_heatmap(fig):
    scene = vector_scene(fig)
    # Set the file or directory path.
    scene.update(pdf=str(ROOT), png=str(ROOT), dpi=PNG_DPI)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    app = com.Dispatch("Illustrator.Application.29")
    with tempfile.TemporaryDirectory(prefix="illustrator-", dir=TMP_DIR) as td:
        # Set the file or directory path.
        script = ROOT
        script.write_text("var S=" + json.dumps(scene, ensure_ascii=True, separators=(",", ":")) + ";\n" + ILLUSTRATOR_JS, encoding="utf-8")
        report = json.loads(app.DoJavaScriptFile(str(script)))
    assert [x["name"] for x in report["layers"]] == ["Axes", "Text", "Legend", "Graphics"]
    assert all(x["groups"] == x["masks"] == 1 for x in report["layers"])
    assert report["clipping_groups"] == report["clipping_paths"] == 4
    assert report["raster_items"] == report["placed_items"] == 0
    assert sorted(report["texts"]) == sorted(x["text"] for x in scene["texts"])
    assert set(report["fonts"]) <= {"ArialMT", "Arial-BoldMT", "Arial-ItalicMT"}
    with PILImage.open(scene["png"]) as image:
        image.load(); image = image.copy()
    expected_size = tuple(round(v * PNG_DPI / 72) for v in scene["size"])
    assert max(abs(a - b) for a, b in zip(image.size, expected_size)) <= 4  # Illustrator rounds artboard bounds to pixels.
    image.save(scene["png"], dpi=(PNG_DPI, PNG_DPI))
    plt.close(fig)
    print("Illustrator PDF verified: 4 layers, 4 masks,",len(report["texts"]),"complete Arial text frames, 0 raster objects.")
    print("PNG:",image.size,"at",PNG_DPI,"DPI")
    return report

## Export and preview

Save the PDF and PNG without changing source RNA. The heatmap H5AD supports redrawing without rerunning upstream statistics.


In [ ]:
#8. =========== Save the editable PDF and 600-DPI PNG
fig = make_heatmap_layout(summary)
export_report = export_editable_heatmap(fig)
# Set the file or directory path.
display(Image(filename=str(ROOT), width=950))